James Caldwell <br>
October 2025 <br>

This script automates prize emails for IGOW/RaceGOW

In [4]:
import numpy as np
import pandas as pd
import tkinter as tk
from tkinter import ttk
import pandas as pd
import time
from datetime import datetime
import re

In [9]:
def url_to_dataframe(sheet_url):    
    try:
        pattern = r"https://docs\.google\.com/spreadsheets/d/([a-zA-Z0-9_-]+)/edit\?gid=(\d+)"
        match = re.search(pattern, sheet_url)
        sheet_id, gid = match.groups()
        csv_url = f"https://docs.google.com/spreadsheets/d/{sheet_id}/export?format=csv&gid={gid}"
        df = pd.read_csv(csv_url, skiprows=1)
        # print("Sheet ID:", sheet_id)
        # print("GID:", gid)
    except Exception as e:
        print("Error accessing URL:", e)
    return df

In [14]:
links_df = pd.read_csv('links.txt',sep='\t')
links_df.head()

,Variable,Value
0,Registration Sheet link,https://docs.google.com/spreadsheets/d/1jvZho0...
1,Prizes for Everyone,https://docs.google.com/spreadsheets/d/1W3Tl1j...
2,Prizes for Bonus Tier,https://docs.google.com/spreadsheets/d/1W3Tl1j...
3,Emax Bonus Prizes,https://docs.google.com/spreadsheets/d/1W3Tl1j...
4,Velocidrone Prizes,https://docs.google.com/spreadsheets/d/1W3Tl1j...


In [2]:
# Big variables
global competition_name 
competition_name = "RaceGOW5"
registration_email_column_name = "Email" # Column name for email from registration sheet
paypal_email_column_name = "PayPal Email" # Column name for paypal email from registration sheet

In [ ]:


## Download Prize Data from Google Sheets
    # Google sheets full url: https://docs.google.com/spreadsheets/d/1p7wASAe13hgVXKNCimAjb5oJkW2To8OnCojyqOP3WJg/edit?gid=0#gid=0
sheet_id = "1p7wASAe13hgVXKNCimAjb5oJkW2To8OnCojyqOP3WJg"
gid = "0"  # the tab’s unique gid
csv_url = f"https://docs.google.com/spreadsheets/d/{sheet_id}/export?format=csv&gid={gid}"

prize_df = pd.read_csv(csv_url, skiprows=1)
open_prizes = prize_df.iloc[0:87,:] # Only grabs open prizes for now. Need to adjust based on this year's formatting

## Download Emails from Google Sheets
sheet_id = "1jvZho0IOuNjtqPmEF7Gtvi8hAcUeSaiebLWSsAArszg"
gid = "1962900089"  # the tab’s unique gid
csv_url = f"https://docs.google.com/spreadsheets/d/{sheet_id}/export?format=csv&gid={gid}"
registration_df = pd.read_csv(csv_url)
registration_df['Email'] = 'charlottesville.drone@gmail.com'
registration_df['PayPal Email'] = 'charlottesville.drone@gmail.com'
callsign_and_email = registration_df[['What is your Pilot Callsign (Handle)?',registration_email_column_name,paypal_email_column_name]]

In [290]:
# Generate list of weeks for GUI dropdown
week_dropdown_list = sorted(open_prizes.iloc[:, 0].unique())



In [291]:
# Get's the selected week from the prize list and adds the email from registration
def get_winner_info(week):
    # Filter prizes for the week
    week_df = open_prizes[open_prizes.iloc[:, 0] == week].copy()
    
    # Merge with callsign_and_email to get both Email and PayPal Email
    # Assuming callsign_and_email has columns: [callsign, Email, PayPal Email]
    week_df = week_df.merge(
        callsign_and_email,
        left_on=week_df.columns[1],  # winner column in week_df
        right_on=callsign_and_email.columns[0],  # callsign column
        how='left'
    )
    
    # Optional: drop the extra 'key' column from merge if needed
    week_df.drop(columns=[callsign_and_email.columns[0]], inplace=True)
    
    return week_df

# test usage
week = 'Track1'
winner_info = get_winner_info(week)
winner_info

,Track#,Winner,Prize Sponsor,Retail Value,Prize Type,Prize Details,Unnamed: 6,Email,PayPal Email
0,Track1,Lone FPV,ZachC82,$200.00,Giveaway,Gift Card to TinyWhoop.com,NaN,charlottesville.drone@gmail.com,charlottesville.drone@gmail.com
1,Track1,cybaix,Tiny Whoop,$100.00,Giveaway,CASH via PayPal,NaN,charlottesville.drone@gmail.com,charlottesville.drone@gmail.com
2,Track1,Hurricane,VIFLY + CustomFPV + The Prop Popper + Gemfan +...,$99.00,Giveaway,Whoopstor V3 + RaceGOW3 Exclusive 65mm 2-bange...,NaN,charlottesville.drone@gmail.com,charlottesville.drone@gmail.com
3,Track1,GunjaFPV,The Prop Popper + HQProp + Gemfan,$37.00,Giveaway,The Original Prop Popper + HQ and Gemfan 31mm ...,NaN,NaN,NaN
4,Track1,Yamasaur,CustomFPV,$30.00,Giveaway,RaceGOW3 Exclusive 65mm 2-banger Whoop Box,NaN,charlottesville.drone@gmail.com,charlottesville.drone@gmail.com
5,Track1,FpvGaps,CustomFPV,$30.00,Giveaway,RaceGOW3 Exclusive 65mm 2-banger Whoop Box,NaN,NaN,NaN
6,Track1,JANxD,Tiny Whoop,$25.00,Giveaway,CASH via PayPal,NaN,NaN,NaN
7,Track1,GOONZ,FPVSkittles + Gemfan,$17.00,Giveaway,Mystery Draw + Gemfan Props and Extras,NaN,charlottesville.drone@gmail.com,charlottesville.drone@gmail.com
8,Track1,Rodgsilva,Nick Burns,$20.00,Livestream,Must be present during livestream draw to win!...,NaN,NaN,NaN


https://developers.google.com/workspace/docs/api/quickstart/python#enable_the_api

Make new project (i called mine IGOW)
-> second tab then click "enable api"

Desktop client 1

download json file and save to working directory



add self to tester list under "audience"


enable gmail api

In [293]:
# This section sets up the Gmail api settings
# This section has no gui associated with it

from __future__ import print_function
import os.path
import base64
from email.mime.text import MIMEText
from google.auth.transport.requests import Request
from google.oauth2.credentials import Credentials
from google_auth_oauthlib.flow import InstalledAppFlow
from googleapiclient.discovery import build

# If modifying scopes, delete the file token.json.
SCOPES = ['https://www.googleapis.com/auth/gmail.send']

def gmail_service_setup():
    """Authenticate and return a Gmail API service."""
    creds = None
    # token.json stores user’s access/refresh tokens after first login
    if os.path.exists('token.json'):
        creds = Credentials.from_authorized_user_file('token.json', SCOPES)
    # If no valid credentials, prompt user login
    if not creds or not creds.valid:
        if creds and creds.expired and creds.refresh_token:
            creds.refresh(Request())
        else:
            flow = InstalledAppFlow.from_client_secrets_file(
                'credentials.json', SCOPES)
            creds = flow.run_local_server(port=0)
        # Save the credentials for next time
        with open('token.json', 'w') as token:
            token.write(creds.to_json())
    return build('gmail', 'v1', credentials=creds)

def create_message(sender, to, subject, message_text):
    """Create a MIMEText email and encode in base64."""
    message = MIMEText(message_text)
    message['to'] = to
    message['from'] = sender
    message['subject'] = subject
    raw = base64.urlsafe_b64encode(message.as_bytes())
    return {'raw': raw.decode()}

def send_message(service, user_id, message):
    """Send an email via Gmail API."""
    sent_message = service.users().messages().send(userId=user_id, body=message).execute()
    # print(f"Message Id: {sent_message['id']}") # message id, uncomment for troubleshooting
    return sent_message



In [294]:
def log_email_sent(message): 
    with open("email_log.txt", "a") as log_file: 
        log_file.write(message + "\n")


def safe_str(x):
    """Convert any value to a safe string (replace NaN with empty)."""
    try:
        if pd.isna(x):
            return ""
    except TypeError:
        pass
    return str(x)

In [ ]:
# Create main window
root = tk.Tk()
root.state("zoomed")
root.title("IGOW/RaceGOW Prize Automation App")

# Instruction Label
instruction_label = tk.Label(
    root,
    text="1. Select a track week from the drop down \n2. Click generate and confirm button \n3. Confirm prizes/emails  \n4. Click Run to send winner emails.",
    wraplength=350,     # wrap text after 350 pixels
    justify="left",     # align text to the left
    # fg="gray"           # grey color for subtle look
)
instruction_label.pack(pady=(5, 10))

# ===== TESTING CHECKBOX + DROPDOWN SECTION =====
frame = tk.Frame(root)
frame.pack(pady=5)
tk.Label(
    frame,
    text="------------ TESTING ---------- \nLeave unchecked to send all emails to email below. "
         "Check to send actual prize emails."
).pack(side="left", padx=5)
test_checkbox_var = tk.IntVar(value=0)
checkbox = tk.Checkbutton(frame, variable=test_checkbox_var)
checkbox.pack(side="left", padx=5)
# --- Testing dropdown ---
tk.Label(root, text="TESTING email: all emails will be sent here instead of actual winner emails (if box above is unchecked)").pack(pady=5)
test_email_var = tk.StringVar(value='charlottesville.drone@gmail.com')  # default value
test_dropdown = ttk.Combobox(
    root,
    textvariable=test_email_var,
    values=['charlottesville.drone@gmail.com', 'igowhoop@gmail.com'],
    state="readonly"
)
test_dropdown.pack(pady=5)

# ===== TRACK SELECTION DROPDOWN =====
tk.Label(root, text="------------------------------\n\nSelect track #:").pack(pady=5)
# week_dropdown_list = ["Week 1", "Week 2", "Week 3"]  # placeholder list
track_week_var = tk.StringVar()
track_dropdown = ttk.Combobox(
    root,
    textvariable=track_week_var,
    values=week_dropdown_list,
    state="readonly"
)
track_dropdown.pack(pady=5)
track_dropdown.current(2)  # ✅ sets the default selection (first item)

# ===== Output Label =====
output_label = tk.Label(root, text="", fg="blue")
output_label.pack(pady=5)

# ===== Button =====
def generate_table():
    # Clear previous rows
    for row in tree.get_children():
        tree.delete(row)
    
    selected_week = track_week_var.get()
    winner_info = get_winner_info(selected_week)
    global winner_abbreviated
    winner_abbreviated = winner_info[['Winner','Prize Details','Prize Sponsor','Email','PayPal Email']]
    
    # Insert rows into Treeview
    for _, row in winner_abbreviated.iterrows():
        tree.insert("", tk.END, values=list(row))
    
    output_label.config(text=f"Generating emails for {selected_week} winners:")

# ===== Button =====
submit_button = tk.Button(root, text="Generate prize table and emails to confirm before sending", command=generate_table)
submit_button.pack(pady=10)

# ===== Treeview for Table =====
columns = ("Winner", "Prize Details","Prize Sponsor", "Email", "PayPal Email")
tree = ttk.Treeview(root, columns=columns, show="headings", height=10)
for col in columns:
    tree.heading(col, text=col)
    tree.column(col, width=200)  # adjust width as needed
tree.pack(pady=5, fill=tk.X)

# Add vertical scrollbar
scrollbar = ttk.Scrollbar(root, orient="vertical", command=tree.yview)
tree.configure(yscrollcommand=scrollbar.set)
scrollbar.pack(side=tk.RIGHT, fill=tk.Y)

# ===== Output Label =====
output_label2 = tk.Label(root, text="", fg="blue")
output_label2.pack(pady=5)

# ===== Button =====
def on_submit():

    output_label2.config(text=f"Sending emails...")

    root.after(500, send_email) # wait 500ms before calling send_email to allow label update

def send_email():
    
    # Start log with date and time:
    timestamp = datetime.now().strftime("%Y-%m-%d %H:%M")
    log_email_sent('----'+timestamp+'----')

    for idx, row in winner_abbreviated.iterrows():
        if test_checkbox_var.get():
            to_email = row['Email']
        else:
            to_email = test_email_var.get()

        to_email = safe_str(to_email)
        print(to_email)

        winner = row['Winner']
        prize = row['Prize Details']
        prize_sponsor = row['Prize Sponsor']
        paypal_email = row['PayPal Email']

        if 'cash' in prize.lower(): # Paypal cash prize
            prize_or_paypal_message = f'Please reply and confirm your PayPal email and I will send the prize: {paypal_email}'
        else:
            prize_or_paypal_message = 'Please reply with your shipping address and information for your prize to be sent to you.'
        # print(prize_or_paypal_message)

        service = gmail_service_setup()
        try:
            email_msg = create_message(
                sender="charlottesville.drone@gmail.com",
                to=to_email,
                subject=f"Claim Your {competition_name} Prize!",
            message_text = (
                f"Congratulations {winner}!\n\n"
                f"You won a '{prize}' in {competition_name} sponsored by: {prize_sponsor}\n\n"
                f"{prize_or_paypal_message}\n\n"
                f"Thanks for participating in {competition_name} and good luck in the rest of the game!"
                + (f"\n\nTESTING condition, email would have gone to: {row['Email']}" if not test_checkbox_var.get() else "" )
            )
                )
            send_message(service, "me", email_msg)
            log_message = str(idx) + '. Email sent to ' + winner + ' / ' + to_email + ' / ' + prize
            
            print(log_message)
        except Exception as err:
            print(err)
            # log_message = "Failed to send email to " + winner + " with email: " + to_email + ":"  + str(e) + " prize: " + prize
            log_message = (
                str(idx) +
                ". Failed to send email to " + str(winner)
                + " with email: " + str(to_email)
                + ": " + str(err)
                + " prize: " + str(prize)
            )
            # print(f"Failed to send email to") # {winner} with email: {to_email}: {e}") # for prize {prize}
        log_email_sent(log_message)
        
        if test_checkbox_var.get():
            output_label2.config(text=f"Sending emails...{winner}")
        else:
            output_label2.config(text=f"Sending to testing email {to_email} for winner {winner}")
        time.sleep(0.25)
        root.update()
    time.sleep(0.25)
    output_label2.config(text=f"Emails sent!")

# ===== Button =====
submit_button = tk.Button(root, text="Send emails", command=on_submit)
submit_button.pack(pady=10)

# ===== Close handler =====
def on_close():
    root.quit()
    root.destroy()

root.protocol("WM_DELETE_WINDOW", on_close)
root.mainloop()


charlottesville.drone@gmail.com
0. Email sent to Lone FPV / charlottesville.drone@gmail.com / Gift Card to TinyWhoop.com
charlottesville.drone@gmail.com
1. Email sent to cybaix / charlottesville.drone@gmail.com / CASH via PayPal
charlottesville.drone@gmail.com
2. Email sent to Hurricane / charlottesville.drone@gmail.com / Whoopstor V3 + RaceGOW3 Exclusive 65mm 2-banger Whoop Box + Prop Popper + HQ and Gemfan 31mm Props
charlottesville.drone@gmail.com
3. Email sent to GunjaFPV / charlottesville.drone@gmail.com / The Original Prop Popper + HQ and Gemfan 31mm Props
charlottesville.drone@gmail.com
4. Email sent to Yamasaur / charlottesville.drone@gmail.com / RaceGOW3 Exclusive 65mm 2-banger Whoop Box
charlottesville.drone@gmail.com
5. Email sent to FpvGaps / charlottesville.drone@gmail.com / RaceGOW3 Exclusive 65mm 2-banger Whoop Box
charlottesville.drone@gmail.com
6. Email sent to JANxD / charlottesville.drone@gmail.com / CASH via PayPal
charlottesville.drone@gmail.com
7. Email sent to GO